In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [ ]:
dataset = load_dataset("spider")

model_path = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are an expert SQL assistant. Given a natural language question and a database schema, generate the correct SQL query."
        },
        {
            "role": "user",
            "content": f"Database: {example['db_id']}\nQuestion: {example['question']}"
        },
        {
            "role": "assistant",
            "content": example['query']
        }
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_data = dataset["train"].map(format_example)
val_data   = dataset["validation"].map(format_example)

In [3]:
if torch.cuda.is_available():
    use_bf16 = torch.cuda.is_bf16_supported()
    use_fp16 = not use_bf16
    use_cpu  = False
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    print(f"Usando GPU: {torch.cuda.get_device_name(0)} | bf16={use_bf16} | fp16={use_fp16}")
else:
    use_bf16 = False
    use_fp16 = False
    use_cpu  = True
    compute_dtype = torch.float32
    print("Usando CPU")

Usando GPU: NVIDIA RTX A4500 | bf16=True | fp16=False


In [ ]:
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.eos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="./qwen-spider-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=40,
    bf16=use_bf16,
    fp16=use_fp16,
    use_cpu=use_cpu,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
    disable_tqdm=False,
    logging_strategy="steps",
    pad_token="<|endoftext|>",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=lora_config,
    args=sft_config,
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/tmp/ipykernel_3049012/3914887879.py:31: FutureWarning: `pad_token` is deprecated and will be removed in v2.0.0. Set `tokenizer.pad_token` directly and pass it as `processing_class` to the trainer instead.
  sft_config = SFTConfig(


In [ ]:
trainer.train()
trainer.save_model("./qwen-spider-finetuned32/r64q4a128")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
200,0.465959,0.743072
400,0.390449,0.778669
600,0.312112,0.813245
800,0.296351,0.839113
1000,0.217843,0.922389
1200,0.219706,0.933373
1314,0.221626,0.933224


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config, 
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, "./qwen-spider-finetuned32/final")

def generate_sql(question, db_id):
    messages = [
        {"role": "system", "content": "You are an expert SQL assistant. Given a natural language question and a database schema, generate the correct SQL query."},
        {"role": "user", "content": f"Database: {db_id}\nQuestion: {question}"}
    ]
    
    # tokenizer separado para ter acesso ao input_ids e attention_mask
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True,  
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,             
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,        
            pad_token_id=tokenizer.eos_token_id,
        )

    # decodifica só os tokens gerados, sem o prompt
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [11]:
print(generate_sql("Find the names of employees and their department managers.", "concert_singer"))

SELECT T1.name ,  T2.manager FROM employee AS T1 JOIN department AS T2 ON T1.department_id  =  T2.id
